In [1]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import optuna

In [2]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATASET_SUBSET = 10000 # Use 10k images for faster trials
NUM_EPOCHS_PER_TRIAL = 3 # Train for fewer epochs during search

In [3]:
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

In [4]:
train_dataset = datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)

val_dataset = datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)

In [5]:
train_subset = Subset(train_dataset, range(DATASET_SUBSET))
val_subset = Subset(val_dataset, range(DATASET_SUBSET // 5))

In [10]:
# The models architecture itself can be a hyperparameter
class SimpleCNN(nn.Module):
    def __init__(self, trial):
        super(SimpleCNN, self).__init__()
        # Suggest hyperparameters for the model architecture
        out_channels_1 = trial.suggest_categorical("out_channels_1", [16, 32])
        out_channels_2 = trial.suggest_categorical("out_channels_2", [32, 64, 128])
        dropout_rate = trial.suggest_float("dropout_rate", 0.2, 0.5)
        
        self.conv_stack = nn.Sequential(
            nn.Conv2d(in_channels = 3, out_channels = out_channels_1, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size = 2, stride = 2),
            nn.Conv2d(in_channels = out_channels_1, out_channels = out_channels_2, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size = 2, stride = 2)
        )
        
        self.fc_stack = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features = out_channels_2 * 8 * 8, out_features = 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(in_features = 128, out_features = 10)
        )
            
    def forward(self, x):
        x = self.conv_stack(x)
        x = self.fc_stack(x)
        return x


In [7]:
# Objective function for optuna
def objective(trial):
    # Suggest Hyperparameters for the training process
    model = SimpleCNN(trial).to(DEVICE)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log = True)
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "RMSprop", "SGD"])
    batch_size = trial.suggest_categorical("batch_size", [32, 64])
    
    # Setup DataLoader and Optimizer
    train_loader = DataLoader(train_subset, batch_size = batch_size, shuffle = True)
    val_loader = DataLoader(val_subset, batch_size = batch_size, shuffle = False)
    optimizer = getattr(torch.optim, optimizer_name)(model.parameters(), lr = lr)
    criterion = nn.CrossEntropyLoss()
    
    # Training and Validation
    for epoch in range(NUM_EPOCHS_PER_TRIAL):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
    model.eval()
    n_correct = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            n_correct += (predicted == labels).sum().item()
            
    accuracy = n_correct / len(val_loader.dataset)
    
    # Optuna will try to maximize this value
    return accuracy

In [11]:
study = optuna.create_study(direction = "maximize")
study.optimize(func = objective, n_trials = 30)

[I 2025-08-07 18:39:33,309] A new study created in memory with name: no-name-76eb4943-ece3-4f28-bdec-18516358e8cd
[I 2025-08-07 18:39:45,304] Trial 0 finished with value: 0.4775 and parameters: {'out_channels_1': 32, 'out_channels_2': 64, 'dropout_rate': 0.49317737856932864, 'lr': 0.0001702053674231137, 'optimizer': 'RMSprop', 'batch_size': 32}. Best is trial 0 with value: 0.4775.
[I 2025-08-07 18:39:51,850] Trial 1 finished with value: 0.4505 and parameters: {'out_channels_1': 16, 'out_channels_2': 64, 'dropout_rate': 0.2960054955954072, 'lr': 0.0001510093177381866, 'optimizer': 'Adam', 'batch_size': 64}. Best is trial 0 with value: 0.4775.
[I 2025-08-07 18:39:57,796] Trial 2 finished with value: 0.402 and parameters: {'out_channels_1': 16, 'out_channels_2': 32, 'dropout_rate': 0.4467191971035131, 'lr': 0.00012049977464116253, 'optimizer': 'RMSprop', 'batch_size': 64}. Best is trial 0 with value: 0.4775.
[I 2025-08-07 18:40:03,535] Trial 3 finished with value: 0.2155 and parameters: {

In [12]:
# --- 6. Print the Results ---
print("\n--- Hyperparameter Search Finished ---")
print(f"Number of finished trials: {len(study.trials)}")

print("\nBest trial:")
trial = study.best_trial
print(f"  Value (Validation Accuracy): {trial.value:.4f}")

print("\n  Best Parameters:")
for key, value in trial.params.items():
    print(f"    {key}: {value}")


--- Hyperparameter Search Finished ---
Number of finished trials: 30

Best trial:
  Value (Validation Accuracy): 0.5740

  Best Parameters:
    out_channels_1: 32
    out_channels_2: 64
    dropout_rate: 0.33639561777109617
    lr: 0.00215120768866737
    optimizer: Adam
    batch_size: 64
